In [88]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [89]:
N_SIM = 10000
n_periods = 20
mu_ibap_price,sigma_ibap_price = 1.6467,0.035
mu_HF_price,sigma_HF_price = 0.0198, 0.0876
mu_HF_throughput,sigma_HF_throughput = 294480, 121.34
mu_ibap_throughput,sigma_ibap_throughput = 578880,170.29
mu_raw_mat,sigma_raw_mat = 7274130,727413
mu_inflation,sigma_inflation = 2,2.25



# Stochastic Drivers (Vectorised for speed)
#  draw all samples at once for each driver
price_of_ibap = np.random.normal(loc=mu_ibap_price, scale=sigma_ibap_price, size=(N_SIM, n_periods))
price_of_ibap = np.exp(price_of_ibap)  # Convert log-price to price

price_of_HF = np.random.normal(loc=mu_HF_price, scale=sigma_HF_price, size=(N_SIM, n_periods))
price_of_HF = np.exp(price_of_HF)  # Convert log-price to price

HF_throughput = np.random.normal(loc=mu_HF_throughput, scale=sigma_HF_throughput, size=(N_SIM, n_periods))
ibap_throughput = np.random.normal(loc=mu_ibap_throughput, scale=sigma_ibap_throughput, size=(N_SIM, n_periods))
raw_material_cost = np.random.normal(loc=mu_raw_mat, scale=sigma_raw_mat, size=(N_SIM, n_periods))
inflation = np.random.normal(loc=mu_inflation, scale=sigma_inflation, size=(N_SIM, n_periods))


In [90]:
print(price_of_ibap[:5,:20])
print(price_of_HF[:5,:20])

[[5.35115076 5.01672971 5.27441529 5.39140138 5.0453548  5.41685051
  5.32388767 5.03524581 5.07733744 5.21468179 5.43392452 5.27286886
  4.98106343 5.23036196 5.27375588 4.94325271 5.17681398 5.22638834
  5.26770855 5.727629  ]
 [5.09310646 4.86756352 5.38295316 5.14893772 5.27808562 5.14443962
  5.20700718 5.00915947 5.12627542 5.27895105 5.2705597  5.32826348
  5.31282657 4.95031945 5.36229851 4.71063046 5.35902232 5.06622056
  5.01333821 5.27813119]
 [5.24630708 5.46825794 5.22674751 4.9939191  5.18699574 5.09915124
  5.28609565 4.98495267 5.01557299 5.04647042 5.57481787 5.24120031
  5.16520036 5.42419163 5.3702199  5.08289728 4.88690572 5.28423818
  5.21029897 4.82649733]
 [5.21481115 5.40485831 5.22377442 5.08542275 4.88089194 5.10430032
  5.03248923 5.08721073 5.28951397 5.25980988 5.0487279  5.47692835
  5.23823878 5.39352685 5.19536135 5.347564   5.18257346 5.03901025
  5.14025915 5.17228536]
 [5.4508515  4.84484406 5.24232487 5.35585983 5.35977743 5.03497857
  5.31560066 5.2

In [91]:
def npv_vectorized(price_of_ibap, ibap_throughput, 
                   price_of_HF,HF_throughput,
                   raw_material_cost, inflation, discount_rate):
    periods = price_of_ibap.shape[1]
    t = np.arange(1, periods + 1)

    revenue = price_of_ibap * ibap_throughput + price_of_HF * HF_throughput
    cost = raw_material_cost * (1 + inflation) ** t
    net_cash_flow = revenue - cost
    discounted_cash_flow = net_cash_flow / (1 + discount_rate) ** t

    return discounted_cash_flow   # one NPV per simulation


In [92]:
npv_results = npv_vectorized(
    price_of_ibap,
    ibap_throughput,
    price_of_HF,HF_throughput,
    raw_material_cost,
    inflation,
    # Baseline in lecture notes
    discount_rate=0.15
)


In [93]:
npv_results[:2,:20]

array([[-5.54318307e+06, -1.85226652e+08, -6.31353014e+08,
        -3.94506137e+08, -5.34815755e+09, -6.85167208e+10,
        -1.12956967e+08, -3.65115653e+12, -2.32569509e+12,
        -8.05081799e+04,  7.43971408e+05, -4.23273042e+11,
        -1.79722577e+07, -9.87128273e+15, -7.80364484e+14,
        -7.76170890e+11,  3.90242840e+05, -1.25209146e+10,
         2.38475441e+05, -6.37035746e+15],
       [-5.09665239e+06,  1.96811424e+06, -3.10893370e+08,
         1.87757289e+06, -6.59993879e+06, -3.66503109e+10,
        -5.47476529e+12,  1.03402382e+06, -9.87929033e+11,
        -7.61238080e+08, -9.34865318e+12, -1.22126962e+12,
        -1.13658180e+12, -2.44134230e+11, -4.47130985e+11,
        -7.62522633e+09, -1.90661082e+13, -2.52877551e+18,
        -1.90003526e+19, -2.07852009e+12]])

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(npv_results.shape[0]):
    plt.plot(range(1, npv_results.shape[1] + 1), npv_results[i], alpha=0.008, color="steelblue")

# Set y-limits using percentiles to ignore extreme outliers
ymin = np.percentile(npv_results, 5)
ymax = np.percentile(npv_results, 75)
plt.ylim(ymin, ymax)

plt.xlabel("Time Period")
plt.ylabel("Value")
plt.title("Monte Carlo Simulation Paths")
plt.show()